In [ ]:
from pymongo import MongoClient
from pymongo.collection import Collection
from teaching_tools.ab_test.reset import Reset

r = Reset("192.252.53.2")
r.reset_database()


In [ ]:
# Import your libraries here
from statsmodels.stats.contingency_tables import Table2x2
from statsmodels.stats.power import GofChisquarePower
from teaching_tools.ab_test.experiment import Experiment
from country_converter import CountryConverter
from pymongo.collection import Collection
from pymongo import MongoClient
from pprint import PrettyPrinter
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import random
import plotly.express as px
import math
from scipy.stats import norm
import scipy.stats


In [ ]:
host = "192.252.53.2"


In [ ]:
# Create `client`
client = MongoClient(host=host, port=27017)
# Create `db`
db = client["wqu-abtest"]
# Assign `"mscfe-applicants"` collection to `mscfe_app`
mscfe_app = db["mscfe-applicants"]


In [ ]:
# Aggregate applicants by nationality
result = mscfe_app.aggregate(
    [
        {
            "$group": {
                "_id": "$countryISO2",
                "count": {"$count": {}}
            }
        }
    ]
)

# Load nationality aggregation result into DataFrame
df_nationality = (
    pd.DataFrame(result)
    .rename({"_id": "country_iso2"}, axis="columns")
    .sort_values("country_iso2")
    .reset_index(drop=True)
)

# Check result
print("df_nationality type:", type(df_nationality))
print("df_nationality shape:", df_nationality.shape)
df_nationality.head()


In [ ]:
# Instantiate `CountryConverter`
cc = CountryConverter()

# Create `"country_name"` column
df_nationality["country_name"] = cc.convert(
    names=df_nationality["country_iso2"],
    to="name_short"
)

# Create `"country_iso3"` column
df_nationality["country_iso3"] = cc.convert(
    names=df_nationality["country_iso2"],
    to="ISO3"
)

# Check result
print("df_nationality type:", type(df_nationality))
print("df_nationality shape", df_nationality.shape)
df_nationality.head()


In [ ]:
# Create `build_nat_choropleth` function

# Create percentage column
df_nationality["count_pct"] = (
    df_nationality["count"] / df_nationality["count"].sum()
) * 100


def build_nat_choropleth():
    fig = px.choropleth(
        data_frame=df_nationality,
        locations="country_iso3",
        color="count",
        projection="natural earth",
        color_continuous_scale=px.colors.sequential.Oranges,
        title="MScFE Applicants: Nationalities"
    )

    return fig


# Don't delete the code below 👇
nat_fig = build_nat_choropleth()
nat_fig.show()


In [ ]:
class MongoRepository:
    """Repository class for interacting with MongoDB database.

    Parameters
    ----------
    client : `pymongo.MongoClient`
        By default, `MongoClient(host=host, port=27017)`.
    db : str
        By default, `'wqu-abtest'`.
    collection : str
        By default, `'mscfe-applicants'`.

    Attributes
    ----------
    collection : pymongo.collection.Collection
        All data will be extracted from and loaded to this collection.
    """

    # Task 7.5.5: `__init__` method
    def __init__(
        self,
        client=MongoClient(host="192.252.53.2", port=27017),
        db="wqu-abtest",
        collection="mscfe-applicants"
    ):
        self.collection = client[db][collection]

    # Task 7.5.6: `find_by_date` method
    def find_by_date(self, date_string):
        # Find start datetime
        start = pd.to_datetime(date_string, format="%Y-%m-%d")
    
        # Calculate the end datetime
        end = start + pd.DateOffset(days=1)
    
        # Build PyMongo query
        query = {
            "createdAt": {"$gte": start, "$lt": end},
            "admissionsQuiz": "incomplete"
        }
    
        # Send query to collection, get results
        result = self.collection.find(query)
    
        # Put results into list
        observations = list(result)
    
        return observations
    
    # Task 7.5.7: `update_applicants` method
    def update_applicants(self, observations_assigned):
        """Update applicant documents in collection."""
    
        # Initialize counters
        n = 0
        n_modified = 0
    
        # Iterate through applicants
        for doc in observations_assigned:
            result = self.collection.update_one(
                filter={"_id": doc["_id"]},
                update={"$set": doc}
            )
    
            # Must be inside the for loop
            n += result.matched_count
            n_modified += result.modified_count
    
        # Create results
        transaction_result = {"n": n, "nModified": n_modified}
    
        return transaction_result
    
    # Task 7.5.7: `assign_to_groups` method

    def assign_to_groups(self, date_string):
        # Get observations
        observations = self.find_by_date(date_string)

        # Shuffle observations
        random.seed(42)
        random.shuffle(observations)

        # Get halfway index
        idx = len(observations) // 2

        # Assign first half to control group
        for doc in observations[:idx]:
            doc["inExperiment"] = True
            doc["group"] = "no email (c)"

        # Assign second half to treatment group
        for doc in observations[idx:]:
            doc["inExperiment"] = True
            doc["group"] = "email (t)"

        # Update collection
        result = self.update_applicants(observations)

        return result

    # Task 7.5.13: `find_exp_observations` method
    def find_exp_observations(self):
        result = self.collection.find({"inExperiment": True})
        return list(result)


In [ ]:
repo = MongoRepository()
print("repo type:", type(repo))
repo


In [ ]:
# Don't modify the code below, it will help test `find_by_date` method.
submission = repo.find_by_date("2022-06-01")
submission


In [ ]:
# Don't modify the code below, it will help test `assign_to_groups` method.
date = "2022-06-02"
submission = repo.assign_to_groups(date)


In [ ]:
chi_square_power = GofChisquarePower()

group_size = math.ceil(
    chi_square_power.solve_power(
        effect_size=0.5,
        alpha=0.05,
        power=0.8
    )
)

print("Group size:", group_size)
print("Total # of applicants needed:", group_size * 2)


In [ ]:
# Aggregate no-quiz applicants by sign-up date
result = mscfe_app.aggregate(
    [
        {
            "$match": {
                "admissionsQuiz": "incomplete"
            }
        },
        {
            "$group": {
                "_id": {
                    "$dateTrunc": {
                        "date": "$createdAt",
                        "unit": "day"
                    }
                },
                "count": {"$sum": 1}
            }
        }
    ]
)

# Load result into DataFrame / Series
no_quiz_mscfe = (
    pd.DataFrame(result)
    .rename({"_id": "date", "count": "new_users"}, axis="columns")
    .set_index("date")
    .sort_index()
    .squeeze()
)

# Set Series name and index name
no_quiz_mscfe.name = "new_users"
no_quiz_mscfe.index.name = "date"

print("no_quiz type:", type(no_quiz_mscfe))
print("no_quiz shape:", no_quiz_mscfe.shape)
no_quiz_mscfe.head()


In [ ]:
# Calculate mean and standard deviation
mean = no_quiz_mscfe.mean()
std = no_quiz_mscfe.std()

# Print results
print("no_quiz mean:", mean)
print("no_quiz std:", std)


In [ ]:
exp_days = 7
sum_mean = mean * exp_days
sum_std = std * np.sqrt(exp_days)
print("Mean of sum:", sum_mean)
print("Std of sum:", sum_std)


In [ ]:
# Probability of getting group_size * 2 or fewer no-quiz users
prob_65_or_fewer = scipy.stats.norm.cdf(
    group_size * 2,
    loc=sum_mean,
    scale=sum_std
)

# Probability of getting more than group_size * 2
# This means 65+ if group_size * 2 = 64
prob_65_or_greater = 1 - prob_65_or_fewer

print(
    f"Probability of getting 65+ no_quiz in {exp_days} days:",
    round(prob_65_or_greater, 3),
)


In [ ]:
exp = Experiment(repo=client, db="wqu-abtest", collection="mscfe-applicants")
exp.reset_experiment()
result = exp.run_experiment(days=exp_days, assignment=True)
print("result type:", type(result))
result


In [ ]:
# Don't modify the code below, it will help test `find_exp_observations` method
submission = repo.find_exp_observations()


In [ ]:
result = repo.find_exp_observations()

# Load observations into DataFrame
df = pd.DataFrame(result)

print("df type:", type(df))
print("df shape:", df.shape)
df.head()


In [ ]:
# Create crosstab of group by admissions quiz completion
data = pd.crosstab(
    index=df["group"],
    columns=df["admissionsQuiz"]
)

print("data type:", type(data))
print("data shape:", data.shape)
data


In [ ]:
# Create `build_contingency_bar` function
def build_contingency_bar():
    # Create side-by-side bar chart
    fig = px.bar(
        data_frame=data,
        barmode="group",
        title="MScFE: Admissions Quiz Completion by Group"
    )

    # Set axis labels
    fig.update_layout(
        xaxis_title="Group",
        yaxis_title="Frequency [count]",
        legend={"title": "Admissions Quiz"}
    )

    return fig


# Don't delete the code below 👇
cb_fig = build_contingency_bar()
cb_fig.show()


In [ ]:
contingency_table = Table2x2(data.values)

print("contingency_table type:", type(contingency_table))
contingency_table.table_orig


In [ ]:
chi_square_test = contingency_table.test_nominal_association()

print("chi_square_test type:", type(chi_square_test))
print(chi_square_test)


In [ ]:
odds_ratio = contingency_table.oddsratio.round(1)
print("Odds ratio:", odds_ratio)
